# GTEx tissue feature importance analysis (K-means + SHAP)

This notebook identifies the most important Latent Variables (LVs) from CLAMPfull_BP for each GTEx tissue
using the k-means clustering labels as the target for SHAP explainability.

1. Load GTEx LV data, tissue metadata, and the saved K-means model
2. Use K-means cluster labels as the target variable
3. Train a multiclass Random Forest classifier to predict cluster membership
4. Compute SHAP values to explain which LVs drive each cluster
5. Map clusters to tissues for biological interpretation

💡 **Environment:** `clamp-analyses`

In [ ]:
import os
import pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr
from collections import defaultdict
from pyprojroot.here import here
import shap

import rpy2.robjects as ro
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import pandas2ri
readRDS = ro.r["readRDS"]

import csv
import seaborn as sns

from itertools import groupby

import re
import textwrap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

np.random.seed(42)

# Settings

In [ ]:
GLOBAL_SEED = 42
N_TOP_FEATURES = 10
CORRELATION_THRESHOLD = 0.8

# random forest hyperparameter grid
PARAM_GRID = {
    'n_estimators': [100, 300, 500],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
}

# Output

In [ ]:
# output paths
OUTPUT_DIR = here('output/gtex_feature_importance_kmeans_shap')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")

# Load data

In [ ]:
# load .rds files from CLAMP with different priors
gtex_GO_BP_CLAMP_rds = readRDS(str(here('output/gtex/gtex_C2CP_CLAMP.rds')))


# helper function to extract B matrix and convert to pandas DataFrame
def extract_B_matrix(rds_obj):
    B_matrix = rds_obj.rx2("B")
    with localconverter(ro.default_converter + pandas2ri.converter):
        B_values = ro.conversion.rpy2py(B_matrix)
    df = pd.DataFrame(
        data=B_values,
        index=B_matrix.rownames if B_matrix.rownames else None,
        columns=B_matrix.colnames if B_matrix.colnames else None,
    )
    return df

# extract B matrices
gtex_GO_BP_CLAMP = extract_B_matrix(gtex_GO_BP_CLAMP_rds)

lv_data = gtex_GO_BP_CLAMP
lv_data.head()

In [ ]:
def extract_summary_matrix(rds_obj):
    summary_matrix = rds_obj.rx2("summary")
    with localconverter(ro.default_converter + pandas2ri.converter):
        summary_values = ro.conversion.rpy2py(summary_matrix)
    df = pd.DataFrame(
        data=summary_values,
        index=summary_matrix.rownames if summary_matrix.rownames else None,
        columns=summary_matrix.colnames if summary_matrix.colnames else None,
    )
    return df

# extract summary matrices
gtex_GO_BP_summary = extract_summary_matrix(gtex_GO_BP_CLAMP_rds)
gtex_GO_BP_summary["pathway"] = gtex_GO_BP_summary["pathway"].str.replace("C2CP_", "", regex=False)

print(gtex_GO_BP_summary.shape)
gtex_GO_BP_summary.head()

In [ ]:
# load GTEx metadata
path = here('data/gtex/GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt')

gtex_meta = pd.read_csv(
    path,
    sep='\t',
    header=0,
    dtype=str,
    quoting=csv.QUOTE_NONE,
    engine='python',
    comment=None,
    keep_default_na=False,
    on_bad_lines='warn'
)

print(gtex_meta.shape)
gtex_meta.head()

In [ ]:
# transpose to get samples as rows, LVs as columns
lv_matrix = lv_data.T

# filter metadata to only include samples in LV matrix
meta_filtered = gtex_meta[gtex_meta['SAMPID'].isin(lv_matrix.index)].copy()
meta_filtered = meta_filtered.set_index('SAMPID')
meta_filtered = meta_filtered.loc[lv_matrix.index]

# use SMTS (tissue type) as target
tissue_labels = meta_filtered['SMTS']

print(f"Number of samples: {len(lv_matrix)}")
print(f"Number of LVs: {lv_matrix.shape[1]}")
print(f"Number of unique tissues: {tissue_labels.nunique()}")
print(f"\nTissue distribution:")
print(tissue_labels.value_counts().head())

In [ ]:
tissue_labels.head()

In [ ]:
lv_matrix.head()

# Load K-means model

Load the K-means model trained in `00_kmeans_clustering.ipynb` (CLAMP BP only).
Use cluster labels as the target variable for SHAP analysis.

In [ ]:
# load the K-means model saved from 00_kmeans_clustering.ipynb
model_path = here('output/gtex/kmeans_models/gtex_GO_BP_CLAMP_kmeans_model.pkl')

with open(model_path, 'rb') as f:
    model_bundle = pickle.load(f)

km_model = model_bundle['model']
km_scaler = model_bundle['scaler']
best_k = model_bundle['best_k']
best_approach = model_bundle['best_approach']
saved_ari = model_bundle['ari']

print(f"K-means model: k={best_k}, approach={best_approach}, ARI={saved_ari:.4f}")

# apply the same preprocessing and get cluster assignments
if best_approach == 'scaled':
    X_km = km_scaler.transform(lv_matrix.astype(np.float32))
else:
    X_km = lv_matrix.values.astype(np.float32)

cluster_labels = km_model.predict(X_km)

# verify ARI matches
y_true = tissue_labels.values
reproduced_ari = adjusted_rand_score(y_true, cluster_labels)
print(f"Reproduced ARI: {reproduced_ari:.4f} (saved: {saved_ari:.4f})")

# map clusters to tissues (majority vote)
cluster_tissue_map = {}
cluster_tissue_purity = {}

for c in range(best_k):
    mask = cluster_labels == c
    cluster_tissues = tissue_labels[mask]
    dominant_tissue = cluster_tissues.value_counts().index[0]
    purity = cluster_tissues.value_counts().iloc[0] / len(cluster_tissues)
    cluster_tissue_map[c] = dominant_tissue
    cluster_tissue_purity[c] = purity

# cluster summary
cluster_summary = pd.DataFrame({
    'Cluster': range(best_k),
    'Dominant_Tissue': [cluster_tissue_map[c] for c in range(best_k)],
    'Purity': [cluster_tissue_purity[c] for c in range(best_k)],
    'Size': [np.sum(cluster_labels == c) for c in range(best_k)]
}).sort_values('Dominant_Tissue')

print(f"\nCluster summary ({best_k} clusters):")
display(cluster_summary)

In [ ]:
phenoplier_path = here('data/gtex/phenoplier/gtex-gls-summary-phenomexcan.tsv.gz')

phenoplier = pd.read_csv(
    phenoplier_path,
    sep="\t",
    compression="gzip",
    low_memory=False
)

print(phenoplier.shape)
print(phenoplier.columns.tolist())

# LV correlation analysis

Before running feature importance, we need to check for multicollinearity among LVs.
Highly correlated features can affect SHAP importance interpretation.
Pearson correlation

In [ ]:
# correlation matrix
corr_matrix = lv_matrix.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0,
            xticklabels=False, yticklabels=False)
plt.show()

# find highly correlated pairs
high_corr_threshold = 0.9
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > high_corr_threshold:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

print(f"Number of highly correlated pairs (|r| > {high_corr_threshold}): {len(high_corr_pairs)}")

LVs independent

# Feature importance analysis (K-means + SHAP)

In [ ]:
_files_exist = (
    (OUTPUT_DIR / "accuracy_summary.tsv").exists() and
    (OUTPUT_DIR / "all_shap_positive.tsv").exists() and
    (OUTPUT_DIR / "cumulative_importance.tsv").exists()
)
if _files_exist:
    print("results found, skipping feature importance and SHAP computation")
else:
    print("results not found, running feature importance and SHAP computation")

In [ ]:
if not _files_exist:
    global_seed = 42

    param_distributions = {
        'n_estimators': [100, 300, 500, 1000],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
    }

    y = cluster_labels

    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=global_seed)
    outer_test_scores = []
    best_params_list = []

    for outer_fold, (dev_idx, test_idx) in enumerate(outer_cv.split(lv_matrix, y)):
        print(f"  outer fold {outer_fold+1}/5")
        lv_data_dev = lv_matrix.iloc[dev_idx]
        y_dev = y[dev_idx]
        lv_data_test = lv_matrix.iloc[test_idx]
        y_test = y[test_idx]

        inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=global_seed)
        clf = RandomForestClassifier(random_state=global_seed, class_weight='balanced')
        random_search = RandomizedSearchCV(
            estimator=clf,
            param_distributions=param_distributions,
            n_iter=15,
            cv=inner_cv,
            scoring='balanced_accuracy',
            n_jobs=-1,
            random_state=global_seed
        )
        random_search.fit(lv_data_dev, y_dev)
        best_params_list.append(random_search.best_params_)

        final_model_outer = RandomForestClassifier(
            random_state=global_seed,
            class_weight='balanced',
            **random_search.best_params_
        )
        final_model_outer.fit(lv_data_dev, y_dev)

        y_pred = final_model_outer.predict(lv_data_test)
        score = balanced_accuracy_score(y_test, y_pred)
        outer_test_scores.append(score)
        print(f"    balanced accuracy: {score:.4f}")

    best_fold_idx = np.argmax(outer_test_scores)
    final_params = best_params_list[best_fold_idx]

    print(f"\nnested CV balanced accuracy: {np.mean(outer_test_scores):.4f} +/- {np.std(outer_test_scores):.4f}")
    print(f"best params: {final_params}")

    final_model = RandomForestClassifier(
        random_state=global_seed,
        class_weight='balanced',
        **final_params
    )
    final_model.fit(lv_matrix, y)
    print(f"\nfinal model trained on all {len(lv_matrix)} samples with {best_k} cluster classes")

In [ ]:
if not _files_exist:
    explainer = shap.TreeExplainer(final_model, data=lv_matrix, feature_perturbation="interventional")
    shap_explanation = explainer(lv_matrix)
    print(f"SHAP values shape: {shap_explanation.values.shape}")
    print(f"  n_samples={shap_explanation.values.shape[0]}, n_features={shap_explanation.values.shape[1]}, n_classes={shap_explanation.values.shape[2]}")

In [ ]:
if not _files_exist:
    shap_results_list = []
    cumulative_importance_list = []
    accuracy_list = []
    top5_lvs_by_tissue = {}

    (OUTPUT_DIR / "per_tissue").mkdir(parents=True, exist_ok=True)

    # group clusters by dominant tissue
    tissue_to_clusters = defaultdict(list)
    for c, tissue in cluster_tissue_map.items():
        tissue_to_clusters[tissue].append(c)

    tissues = sorted(tissue_to_clusters.keys())

    for tissue in tissues:
        tissue_clusters = tissue_to_clusters[tissue]
        mean_purity = np.mean([cluster_tissue_purity[c] for c in tissue_clusters])

        all_shap_for_tissue = []
        for c in tissue_clusters:
            cluster_mask = cluster_labels == c
            shap_values_cluster = shap_explanation.values[cluster_mask, :, c]
            all_shap_for_tissue.append(shap_values_cluster)

        shap_values_tissue = np.concatenate(all_shap_for_tissue, axis=0)
        mean_shap_tissue = np.mean(shap_values_tissue, axis=0)

        df_shap_all = pd.DataFrame({
            "Feature": lv_matrix.columns,
            "Mean_SHAP_Tissue": mean_shap_tissue
        })

        df_positive = df_shap_all[df_shap_all["Mean_SHAP_Tissue"] > 0].copy()
        df_positive = df_positive.sort_values("Mean_SHAP_Tissue", ascending=False).reset_index(drop=True)

        # cumulative importance
        n_features_needed = {}
        if len(df_positive) > 0:
            total_positive_shap = df_positive["Mean_SHAP_Tissue"].sum()
            df_positive["Cumulative_SHAP"] = df_positive["Mean_SHAP_Tissue"].cumsum()
            df_positive["Cumulative_Percent"] = (df_positive["Cumulative_SHAP"] / total_positive_shap) * 100
            df_positive["Rank"] = range(1, len(df_positive) + 1)

            for thresh in [50, 70, 80, 90, 95]:
                n_features_needed[thresh] = (df_positive["Cumulative_Percent"] >= thresh).idxmax() + 1

            df_cumulative = df_positive[["Feature", "Mean_SHAP_Tissue", "Cumulative_Percent", "Rank"]].copy()
            df_cumulative["Tissue"] = tissue
            cumulative_importance_list.append(df_cumulative)

        accuracy_list.append({
            "Tissue": tissue,
            "N_Clusters": len(tissue_clusters),
            "Mean_Purity": mean_purity,
            "N_Positive_LVs": len(df_positive),
            "LVs_for_80pct": n_features_needed.get(80),
            "LVs_for_90pct": n_features_needed.get(90),
        })

        df_all_positive = df_positive.copy()
        df_all_positive["Tissue"] = tissue
        shap_results_list.append(df_all_positive)

        safe_name = tissue.replace(' ', '_').replace('/', '_')
        df_positive.to_csv(OUTPUT_DIR / "per_tissue" / f"shap_positive_{safe_name}.tsv", sep="\t", index=False)
        top5_lvs_by_tissue[tissue] = df_positive["Feature"].head(5).tolist()
        print(f"  {tissue}: {len(tissue_clusters)} clusters, purity={mean_purity:.2f}, top LVs={top5_lvs_by_tissue[tissue]}")

    # compile and save results
    accuracy_df = pd.DataFrame(accuracy_list)
    shap_results_df = pd.concat(shap_results_list, ignore_index=True)
    cumulative_df = pd.concat(cumulative_importance_list, ignore_index=True)

    accuracy_df.to_csv(OUTPUT_DIR / "accuracy_summary.tsv", sep="\t", index=False)
    shap_results_df.to_csv(OUTPUT_DIR / "all_shap_positive.tsv", sep="\t", index=False)
    cumulative_df.to_csv(OUTPUT_DIR / "cumulative_importance.tsv", sep="\t", index=False)
    print(f"\ndone. results saved to: {OUTPUT_DIR}")

In [ ]:
accuracy_df = pd.read_csv(OUTPUT_DIR / "accuracy_summary.tsv", sep="\t")
shap_results_df = pd.read_csv(OUTPUT_DIR / "all_shap_positive.tsv", sep="\t")
cumulative_df = pd.read_csv(OUTPUT_DIR / "cumulative_importance.tsv", sep="\t")

In [ ]:
accuracy_df.sort_values(by='Mean_Purity', ascending=False)

In [ ]:
cumulative_df.head(40)

# LV tissue annotation

For each LV, take the top 1% of samples (highest B matrix values) and annotate the LV with the dominant tissue.
Then build a heatmap showing how tissue-specific the top SHAP LVs are.

In [ ]:
# annotate each LV using the top 1% of samples from the B matrix
n_top = max(1, int(len(lv_matrix) * 0.01))

lv_tissue_pct = {}
lv_tissue_annotation = {}

for lv in lv_matrix.columns:
    top_idx = lv_matrix[lv].nlargest(n_top).index
    top_tissues = tissue_labels.loc[top_idx]
    pct_dist = (top_tissues.value_counts(normalize=True) * 100).to_dict()
    lv_tissue_pct[lv] = pct_dist
    lv_tissue_annotation[lv] = top_tissues.value_counts().index[0]

lv_annot_df = pd.DataFrame({
    'LV': list(lv_tissue_annotation.keys()),
    'Annotated_Tissue': list(lv_tissue_annotation.values()),
}).set_index('LV')

print(f"top 1% = {n_top} samples per LV")
print(f"\nLV annotations per tissue:")
print(lv_annot_df['Annotated_Tissue'].value_counts())
lv_annot_df.head(10)

In [ ]:
CUMULATIVE_PCT = 60

valid_tissues = accuracy_df[accuracy_df['Mean_Purity'] >= 0.7]['Tissue'].tolist()
tissues_ordered = sorted(valid_tissues)

results = []
for tissue in tissues_ordered:
    df_t = cumulative_df[cumulative_df['Tissue'] == tissue].sort_values('Rank')
    mask = df_t['Cumulative_Percent'] >= CUMULATIVE_PCT
    n = int(df_t.loc[mask, 'Rank'].iloc[0]) if mask.any() else len(df_t)
    cum_at_n = df_t[df_t['Rank'] == n]['Cumulative_Percent'].values[0]
    results.append({'Tissue': tissue, 'N_lvs': n, 'CumPct': round(cum_at_n, 1)})

results_df = pd.DataFrame(results)
display(results_df)

n_per_tissue = results_df.set_index('Tissue')['N_lvs'].to_dict()
top_per_tissue = {
    tissue: shap_results_df[shap_results_df['Tissue'] == tissue]['Feature']
                .head(n_per_tissue[tissue]).tolist()
    for tissue in n_per_tissue
}

for tissue, n in sorted(n_per_tissue.items()):
    print(f"  {tissue}: {n} LVs")

In [ ]:
TOP_PCT = 0.01  # fraction of samples to use from B matrix for LV annotation

n_top = max(1, int(len(lv_matrix) * TOP_PCT))
heatmap_matrix = np.zeros((len(tissues_ordered), len(tissues_ordered)))

for j, col_tissue in enumerate(tissues_ordered):
    top_lvs = top_per_tissue.get(col_tissue, [])
    pooled_tissues = []
    for lv in top_lvs:
        top_idx = lv_matrix[lv].nlargest(n_top).index
        pooled_tissues.extend(tissue_labels.loc[top_idx].tolist())
    tissue_counts = pd.Series(pooled_tissues).value_counts()
    for i, row_tissue in enumerate(tissues_ordered):
        heatmap_matrix[i, j] = tissue_counts.get(row_tissue, 0)

row_sums = heatmap_matrix.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1
heatmap_matrix_norm = heatmap_matrix / row_sums * 100

heatmap_df = pd.DataFrame(heatmap_matrix_norm, index=tissues_ordered, columns=tissues_ordered)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    heatmap_df,
    cmap='YlGn',
    vmin=0, vmax=100,
    linewidths=0.3,
    linecolor='#f0f0f0',
    ax=ax,
    annot=True,
    fmt='.1f',
    annot_kws={'size': 7},
    cbar_kws={'label': '% of samples (row-normalised)', 'shrink': 0.8}
)
ax.set_xlabel('Tissue (by Top SHAP LVs)', fontsize=12)
ax.set_ylabel('Tissue (GTEx metadata)', fontsize=12)
plt.xticks(rotation=90, fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

## Traits

In [ ]:
phenoplier.head()

In [ ]:
phenoplier_filtered = (
    phenoplier.loc[phenoplier["fdr"] < 0.001, ["phenotype", "phenotype_desc", "lv", "fdr"]]
    .sort_values("fdr")
    .rename(columns={"phenotype": "pheno_id", "phenotype_desc": "phenotype", "lv": "LV"})
)

display(phenoplier_filtered.head())
phenoplier_filtered.shape

In [ ]:
import urllib.request

pheno_info_path = here('output/gtex/phenomexcan_simplified_phenotypes_info.tsv.gz')
if not pheno_info_path.exists():
    url = "https://zenodo.org/records/14941353/files/phenomexcan-phenotypes_info.tsv.gz?download=1"
    urllib.request.urlretrieve(url, pheno_info_path)
    print(f"downloaded to {pheno_info_path}")

pheno_info = pd.read_csv(pheno_info_path, sep='\t', compression='gzip', index_col='pheno_id')

# build description → effective n lookup (use n_cases + n_controls for binary traits)
pheno_info['n_eff'] = pheno_info['n'].combine_first(
    pheno_info['n_cases'] + pheno_info['n_controls']
)
desc_to_n = (
    pheno_info.dropna(subset=['n_eff'])
    .groupby('description')['n_eff']
    .max()
    .astype(int)
    .to_dict()
)

print(f"loaded {len(pheno_info)} phenotypes, {len(desc_to_n)} with n info")
pheno_info.head()

In [ ]:
import textwrap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

MIN_N = 10000

trait_records = {}
seen_lvs = set()

for tissue in tissues_ordered:
    for lv in top_per_tissue.get(tissue, []):
        if lv in seen_lvs:
            continue
        matches = phenoplier_filtered[phenoplier_filtered['LV'] == lv].sort_values('fdr')
        found = False
        for _, mrow in matches.iterrows():
            n_val = desc_to_n.get(mrow['phenotype'])
            if n_val is None or n_val < MIN_N:
                continue
            found = True
            trait_records.setdefault(tissue, []).append({
                'trait': mrow['phenotype'],
                'n': n_val,
                'fdr': mrow['fdr'],
                'LV': lv,
            })
        if found:
            seen_lvs.add(lv)

for tissue in trait_records:
    seen_traits = {}
    for r in sorted(trait_records[tissue], key=lambda r: r['fdr']):
        if r['trait'] not in seen_traits:
            seen_traits[r['trait']] = r
    trait_records[tissue] = list(seen_traits.values())[:10]
    trait_records[tissue] = sorted(trait_records[tissue], key=lambda r: r['fdr'], reverse=True)

tissues_with_traits = [t for t in tissues_ordered if t in trait_records]

if not tissues_with_traits:
    print("no traits matched")
else:
    green_cmap = LinearSegmentedColormap.from_list('white_green', ['#ffffff', '#00441b'])

    all_fdr_t = [r['fdr'] for t in tissues_with_traits for r in trait_records[t]]
    neg_log_all = -np.log10(np.array(all_fdr_t).clip(min=1e-10))
    fdr_max = np.ceil(neg_log_all.max())
    x_max = fdr_max + 0.5
    norm = Normalize(vmin=0, vmax=fdr_max)

    all_n = [r['n'] for t in tissues_with_traits for r in trait_records[t]]
    log_n_min = np.log10(max(min(all_n), 1))
    log_n_max = np.log10(max(all_n))

    def n_to_size(n, s_min=10, s_max=120):
        log_n = np.log10(max(n, 1))
        return ((log_n - log_n_min) / (log_n_max - log_n_min + 1e-10)) * (s_max - s_min) + s_min

    height_ratios = [len(trait_records[t]) for t in tissues_with_traits]
    n = len(tissues_with_traits)
    total_rows = sum(height_ratios)

    fig = plt.figure(figsize=(7.5, total_rows * 0.32 + 1.8), facecolor='white')

    gs = gridspec.GridSpec(
        n, 1, figure=fig,
        height_ratios=height_ratios,
        hspace=0.10,
        top=0.97, bottom=0.05,
        left=0.5, right=0.95,
    )

    for idx, tissue in enumerate(tissues_with_traits):
        ax = fig.add_subplot(gs[idx])
        ax.set_facecolor('white')

        trecs = trait_records[tissue]
        traits    = [textwrap.fill(r['trait'], width=55) for r in trecs]
        ns        = np.array([r['n'] for r in trecs])
        fdrs      = np.array([r['fdr'] for r in trecs])
        neg_log_f = -np.log10(fdrs.clip(min=1e-10))
        sizes     = [n_to_size(nn) for nn in ns]
        y_pos     = np.arange(len(traits))

        ax.scatter(
            neg_log_f, y_pos,
            c=neg_log_f, s=sizes,
            cmap=green_cmap, norm=norm,
            edgecolors='#555555', linewidths=0.4, alpha=0.95, zorder=3,
        )

        ax.set_yticks(y_pos)
        ax.set_yticklabels(traits, fontsize=7, ha='right')
        ax.set_xlim(-0.2, x_max)
        ax.set_ylim(-0.5, len(traits) - 0.5)

        fig.text(
            0.10, (gs[idx].get_position(fig).y0 + gs[idx].get_position(fig).y1) / 2,
            tissue,
            fontsize=7.5, fontweight='bold',
            va='center', ha='center',
            rotation=90,
        )

        ax.grid(True, axis='x', linestyle='--', linewidth=0.4, alpha=0.35, color='#bbbbbb', zorder=0)
        ax.grid(True, axis='y', linestyle='--', linewidth=0.4, alpha=0.35, color='#bbbbbb', zorder=0)
        ax.set_axisbelow(True)

        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.5)
            spine.set_color('#888888')

        if idx == n - 1:
            ax.set_xlabel('-Log$_{10}$(FDR)', fontsize=8)
        else:
            ax.tick_params(labelbottom=False)
        ax.tick_params(left=False, which='both', labelsize=7)

    legend_y = 0.985
    cbar_h = 0.012

    fig.text(
        0.40, legend_y + cbar_h / 2,
        '-Log$_{10}$(FDR)',
        fontsize=7, va='center', ha='right',
    )

    cbar_ax = fig.add_axes([0.42, legend_y, 0.22, cbar_h])
    sm = ScalarMappable(cmap=green_cmap, norm=norm)
    sm.set_array([])
    cb = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
    tick_step = max(1, int(fdr_max // 5))
    cb.set_ticks(np.arange(0, fdr_max + 1, tick_step))
    cb.ax.tick_params(labelsize=6.5, top=True, bottom=False,
                      labeltop=True, labelbottom=False, length=2.5)

    legend_ns = [v for v in [10000, 50000, 100000, 500000]
                 if min(all_n) * 0.5 <= v <= max(all_n) * 2]
    if not legend_ns:
        legend_ns = [min(all_n), max(all_n)]
    legend_handles = [
        Line2D([0], [0], marker='o', color='w',
               markerfacecolor='#777777',
               markersize=np.sqrt(n_to_size(v)),
               label=f'{v:,}',
               markeredgecolor='#555555', markeredgewidth=0.4)
        for v in legend_ns
    ]
    fig.legend(
        handles=legend_handles,
        title='n',
        bbox_to_anchor=(0.69, legend_y + cbar_h / 2),
        loc='center left',
        fontsize=6.5, title_fontsize=7,
        frameon=True, framealpha=0.9,
        edgecolor='#cccccc',
        ncol=len(legend_ns),
        labelspacing=0.3,
        handletextpad=0.3,
        borderpad=0.4,
        columnspacing=0.4,
    )

    plt.show()

## BP

In [ ]:
gtex_GO_BP_summary_sub = gtex_GO_BP_summary[(gtex_GO_BP_summary['AUC'] > 0.6) & (gtex_GO_BP_summary['FDR'] < 0.1)]

In [ ]:
import re

def make_pathway_short(name):
    name = re.sub(r'^(BIOCARTA|REACTOME|KEGG|WP|GOBP|GO|HALLMARK|PID)_', '', name, flags=re.IGNORECASE)
    return name.replace('_', ' ').title()

records = {}
seen_lvs = set()

for tissue in tissues_ordered:
    for lv in top_per_tissue.get(tissue, []):
        if lv in seen_lvs:
            continue
        seen_lvs.add(lv)
        matches = gtex_GO_BP_summary_sub[gtex_GO_BP_summary_sub['LV'] == lv].sort_values('FDR')
        for _, row in matches.head(5).iterrows():
            records.setdefault(tissue, []).append({
                'pathway_short': make_pathway_short(str(row['pathway'])),
                'AUC': float(row['AUC']),
                'FDR': float(row['FDR']),
            })

# deduplicate within tissue (keep best AUC per pathway), cap at 15
for tissue in records:
    seen = {}
    for r in sorted(records[tissue], key=lambda r: r['AUC'], reverse=True):
        if r['pathway_short'] not in seen:
            seen[r['pathway_short']] = r
    records[tissue] = list(seen.values())[:15]

In [ ]:
import textwrap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

green_cmap = LinearSegmentedColormap.from_list('white_green', ['#ffffff', '#00441b'])
norm = Normalize(vmin=0, vmax=3)

auc_min, auc_max = 0.6, 1.0


def auc_to_size(auc, s_min=10, s_max=120):
    return ((auc - auc_min) / (auc_max - auc_min)) * (s_max - s_min) + s_min


for tissue in records:
    records[tissue] = sorted(records[tissue], key=lambda r: r['AUC'], reverse=False)

tissues_with_data = [t for t in tissues_ordered if t in records]
height_ratios = [len(records[t]) for t in tissues_with_data]
n = len(tissues_with_data)
total_rows = sum(height_ratios)

fig = plt.figure(figsize=(7.5, total_rows * 0.32 + 1.8), facecolor='white')

gs = gridspec.GridSpec(
    n, 1, figure=fig,
    height_ratios=height_ratios,
    hspace=0.10,
    top=0.93, bottom=0.05,
    left=0.53, right=0.88,
)

for idx, tissue in enumerate(tissues_with_data):
    ax = fig.add_subplot(gs[idx])
    ax.set_facecolor('white')

    tissue_records = records[tissue]
    pathways  = [textwrap.fill(r['pathway_short'], width=55) for r in tissue_records]
    aucs      = np.array([r['AUC'] for r in tissue_records])
    fdrs      = np.array([r['FDR'] for r in tissue_records])
    neg_log_f = np.clip(-np.log10(fdrs.clip(min=1e-10)), 0, 3)
    sizes     = [auc_to_size(a) for a in aucs]
    y_pos     = np.arange(len(pathways))

    ax.scatter(
        aucs, y_pos,
        c=neg_log_f, s=sizes,
        cmap=green_cmap, norm=norm,
        edgecolors='#555555', linewidths=0.4, alpha=0.95, zorder=3,
    )

    ax.set_yticks(y_pos)
    ax.set_yticklabels(pathways, fontsize=7, ha='right')
    ax.set_xlim(0.55, 1.05)
    ax.set_ylim(-0.5, len(pathways) - 0.5)

    fig.text(
        0.10, (gs[idx].get_position(fig).y0 + gs[idx].get_position(fig).y1) / 2,
        tissue,
        fontsize=7.5, fontweight='bold',
        va='center', ha='center',
        rotation=90,
    )

    ax.grid(True, axis='x', linestyle='--', linewidth=0.4, alpha=0.35, color='#bbbbbb', zorder=0)
    ax.grid(True, axis='y', linestyle='--', linewidth=0.4, alpha=0.35, color='#bbbbbb', zorder=0)
    ax.set_axisbelow(True)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.5)
        spine.set_color('#888888')

    if idx == n - 1:
        ax.set_xlabel('AUC', fontsize=8)
    else:
        ax.tick_params(labelbottom=False)
    ax.tick_params(left=False, which='both', labelsize=7)

legend_y = 0.965
cbar_h = 0.012

fig.text(
    0.40, legend_y + cbar_h / 2,
    '-Log$_{10}$(FDR)',
    fontsize=7, va='center', ha='right',
)

cbar_ax = fig.add_axes([0.42, legend_y, 0.22, cbar_h])
sm = ScalarMappable(cmap=green_cmap, norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
cb.set_ticks([0.0, 1.0, 2.0, 3.0])
cb.ax.tick_params(labelsize=6.5, top=True, bottom=False,
                  labeltop=True, labelbottom=False, length=2.5)

legend_handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='#777777',
           markersize=np.sqrt(auc_to_size(v)),
           label=f'{v:.1f}',
           markeredgecolor='#555555', markeredgewidth=0.4)
    for v in [0.7, 0.8, 1.0]
]
fig.legend(
    handles=legend_handles,
    title='AUC',
    bbox_to_anchor=(0.69, legend_y + cbar_h / 2),
    loc='center left',
    fontsize=6.5, title_fontsize=7,
    frameon=True, framealpha=0.9,
    edgecolor='#cccccc',
    ncol=3,
    labelspacing=0.3,
    handletextpad=0.3,
    borderpad=0.4,
    columnspacing=0.4,
)

plt.show()